# Real-Time Delay Prediction Model

This notebook builds a classification model to predict delay severity for NYC subway trains
in real time. The model uses historical delay observations collected from the Railtime GTFS-RT
pipeline, enriched with temporal and contextual features.

**Prerequisites:** This notebook requires at least 3 days of real-time data collection via the
Railtime data pipeline. The delay prediction dataset is built by comparing scheduled arrival
times against actual GTFS-RT timestamps, aggregated per trip/stop.

**Target Classes:**
- `on_time`: Delay < 2 minutes
- `minor`: Delay between 2-5 minutes
- `major`: Delay > 5 minutes

In [ ]:
import awswrangler as wr
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 120

%matplotlib inline

In [ ]:
# ---- Configuration ----
# Replace YOUR_ACCOUNT_ID with your actual AWS account ID
ML_BUCKET = "railtime-ml-YOUR_ACCOUNT_ID"

DELAY_DATASET_PATH = f"s3://{ML_BUCKET}/datasets/delay_prediction/"

# Delay severity thresholds (seconds)
ON_TIME_THRESHOLD = 120    # < 2 minutes
MINOR_THRESHOLD = 300      # 2-5 minutes
# > 5 minutes = major

# Rush hour definitions
AM_RUSH_START, AM_RUSH_END = 7, 9
PM_RUSH_START, PM_RUSH_END = 17, 19

# MTA route groupings for analysis
ROUTE_GROUPS = {
    "1/2/3": ["1", "2", "3"],
    "4/5/6": ["4", "5", "6"],
    "7": ["7"],
    "A/C/E": ["A", "C", "E"],
    "B/D/F/M": ["B", "D", "F", "M"],
    "N/Q/R/W": ["N", "Q", "R", "W"],
    "G": ["G"],
    "J/Z": ["J", "Z"],
    "L": ["L"],
    "S": ["S"],
}

In [ ]:
# ---- Load Data ----
df = wr.s3.read_parquet(DELAY_DATASET_PATH)

print(f"Shape: {df.shape}")
print(f"\nDtypes:")
print(df.dtypes)
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Days of data: {(df['timestamp'].max() - df['timestamp'].min()).days}")
print(f"\nRoutes present: {sorted(df['route_id'].unique())}")
print(f"\nNull counts:")
print(df.isnull().sum())
df.head(10)

## Exploratory Data Analysis

We examine the distribution of delays, how they vary by route, time of day, and day of week.

In [ ]:
# ---- Delay Distribution ----
df["timestamp"] = pd.to_datetime(df["timestamp"])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of delay_seconds
# Clip to reasonable range for visualization
delay_clipped = df["delay_seconds"].clip(-300, 900)

axes[0].hist(
    delay_clipped, bins=100, color="steelblue", edgecolor="none", alpha=0.8
)
axes[0].axvline(x=0, color="green", linestyle="--", linewidth=1.5, label="On schedule")
axes[0].axvline(x=ON_TIME_THRESHOLD, color="orange", linestyle="--", linewidth=1.5, label=f"{ON_TIME_THRESHOLD}s (on-time cutoff)")
axes[0].axvline(x=MINOR_THRESHOLD, color="red", linestyle="--", linewidth=1.5, label=f"{MINOR_THRESHOLD}s (minor/major cutoff)")
axes[0].set_title("Distribution of Delay (seconds)", fontweight="bold")
axes[0].set_xlabel("Delay (seconds)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=9)

# Box plot by route
route_order = df.groupby("route_id")["delay_seconds"].median().sort_values().index
sns.boxplot(
    data=df, x="route_id", y="delay_seconds", order=route_order,
    palette="coolwarm", ax=axes[1], fliersize=0.5,
    showfliers=False  # Hide outliers for cleaner view
)
axes[1].axhline(y=0, color="green", linestyle="--", alpha=0.5)
axes[1].axhline(y=ON_TIME_THRESHOLD, color="orange", linestyle="--", alpha=0.5)
axes[1].axhline(y=MINOR_THRESHOLD, color="red", linestyle="--", alpha=0.5)
axes[1].set_title("Delay Distribution by Route (outliers hidden)", fontweight="bold")
axes[1].set_xlabel("Route")
axes[1].set_ylabel("Delay (seconds)")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\nDelay Statistics:")
print(f"  Mean:   {df['delay_seconds'].mean():.1f}s")
print(f"  Median: {df['delay_seconds'].median():.1f}s")
print(f"  Std:    {df['delay_seconds'].std():.1f}s")
print(f"  95th percentile: {df['delay_seconds'].quantile(0.95):.1f}s")
print(f"  % on time (<{ON_TIME_THRESHOLD}s): {(df['delay_seconds'] < ON_TIME_THRESHOLD).mean() * 100:.1f}%")
print(f"  % major (>{MINOR_THRESHOLD}s): {(df['delay_seconds'] > MINOR_THRESHOLD).mean() * 100:.1f}%")

In [ ]:
# ---- Time-of-Day Analysis: Heatmap ----
df["hour"] = df["timestamp"].dt.hour

# Average delay by route and hour
hourly_route = df.groupby(["route_id", "hour"])["delay_seconds"].mean().reset_index()
heatmap_data = hourly_route.pivot(index="route_id", columns="hour", values="delay_seconds")

# Sort routes by overall mean delay
route_mean_order = df.groupby("route_id")["delay_seconds"].mean().sort_values(ascending=False).index
heatmap_data = heatmap_data.reindex(route_mean_order)

fig, ax = plt.subplots(figsize=(18, 10))

sns.heatmap(
    heatmap_data, cmap="RdYlGn_r", center=ON_TIME_THRESHOLD,
    annot=True, fmt=".0f", linewidths=0.5, linecolor="white",
    ax=ax, cbar_kws={"label": "Mean Delay (seconds)"}
)

# Highlight rush hours
for rush_start, rush_end in [(AM_RUSH_START, AM_RUSH_END), (PM_RUSH_START, PM_RUSH_END)]:
    ax.axvline(x=rush_start, color="red", linewidth=2, alpha=0.5)
    ax.axvline(x=rush_end + 1, color="red", linewidth=2, alpha=0.5)

ax.set_title("Average Delay by Route and Hour (red bands = rush hours)", fontsize=14, fontweight="bold")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Route")

plt.tight_layout()
plt.show()

In [ ]:
# ---- Day-of-Week Patterns ----
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["day_name"] = df["timestamp"].dt.day_name()

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot by day of week
sns.boxplot(
    data=df, x="day_name", y="delay_seconds", order=day_order,
    palette="coolwarm", ax=axes[0], fliersize=0.3, showfliers=False
)
axes[0].axhline(y=ON_TIME_THRESHOLD, color="orange", linestyle="--", alpha=0.5)
axes[0].set_title("Delay Distribution by Day of Week", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Delay (seconds)")
axes[0].tick_params(axis="x", rotation=45)

# Delay severity proportions by day
df["severity_temp"] = pd.cut(
    df["delay_seconds"],
    bins=[-np.inf, ON_TIME_THRESHOLD, MINOR_THRESHOLD, np.inf],
    labels=["on_time", "minor", "major"]
)

severity_by_day = pd.crosstab(
    df["day_name"], df["severity_temp"], normalize="index"
).reindex(day_order) * 100

severity_by_day.plot(
    kind="bar", stacked=True, ax=axes[1],
    color=["#2ecc71", "#f39c12", "#e74c3c"], edgecolor="none"
)
axes[1].set_title("Delay Severity Proportions by Day", fontweight="bold")
axes[1].set_ylabel("Percentage")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(title="Severity", loc="upper right")

plt.tight_layout()
plt.show()

df.drop(columns=["severity_temp"], inplace=True)

## Feature Engineering

We create lag features, rolling statistics, temporal interactions, and contextual flags
to give the model information about recent delay patterns and operational conditions.

In [ ]:
# ---- Create Features ----
feat = df.sort_values(["route_id", "stop_id", "timestamp"]).reset_index(drop=True)

# Lag features: delay at t-1, t-2, t-3 for same route+stop
for lag in [1, 2, 3]:
    feat[f"delay_lag_{lag}"] = feat.groupby(["route_id", "stop_id"])["delay_seconds"].shift(lag)

# Rolling mean of last 3 observations for same route+stop
feat["delay_rolling_mean_3"] = feat.groupby(["route_id", "stop_id"])["delay_seconds"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# Rolling std of last 3 observations
feat["delay_rolling_std_3"] = feat.groupby(["route_id", "stop_id"])["delay_seconds"].transform(
    lambda x: x.rolling(3, min_periods=1).std()
).fillna(0)

# Temporal features
feat["hour"] = feat["timestamp"].dt.hour
feat["day_of_week"] = feat["timestamp"].dt.dayofweek
feat["is_weekend"] = (feat["day_of_week"] >= 5).astype(int)
feat["minute_of_day"] = feat["timestamp"].dt.hour * 60 + feat["timestamp"].dt.minute

# Hour * day_of_week interaction
feat["hour_x_dow"] = feat["hour"] * feat["day_of_week"]

# Rush hour flags
feat["is_am_rush"] = ((feat["hour"] >= AM_RUSH_START) & (feat["hour"] <= AM_RUSH_END)).astype(int)
feat["is_pm_rush"] = ((feat["hour"] >= PM_RUSH_START) & (feat["hour"] <= PM_RUSH_END)).astype(int)
feat["is_rush_hour"] = (feat["is_am_rush"] | feat["is_pm_rush"]).astype(int)

# Alert active flag (if column exists in dataset)
if "alert_active" not in feat.columns:
    # Derive from delay spikes: if route's rolling mean > 300s, likely an alert situation
    route_rolling = feat.groupby("route_id")["delay_seconds"].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )
    feat["alert_active"] = (route_rolling > MINOR_THRESHOLD).astype(int)

# Cyclical encoding for hour
feat["hour_sin"] = np.sin(2 * np.pi * feat["hour"] / 24)
feat["hour_cos"] = np.cos(2 * np.pi * feat["hour"] / 24)

# Encode route_id as numeric
route_encoder = LabelEncoder()
feat["route_encoded"] = route_encoder.fit_transform(feat["route_id"])

# Encode stop_id as numeric
stop_encoder = LabelEncoder()
feat["stop_encoded"] = stop_encoder.fit_transform(feat["stop_id"])

# Drop rows with NaN from lag features
feat_clean = feat.dropna(subset=["delay_lag_1", "delay_lag_2", "delay_lag_3"]).reset_index(drop=True)

print(f"Feature matrix shape: {feat_clean.shape}")
print(f"Features created: {len([c for c in feat_clean.columns if c not in df.columns])}")
print(f"Rows dropped due to lag NaN: {len(feat) - len(feat_clean)}")
feat_clean.head()

In [ ]:
# ---- Create Target Variable ----
def classify_delay(seconds):
    if seconds < ON_TIME_THRESHOLD:
        return 0  # on_time
    elif seconds < MINOR_THRESHOLD:
        return 1  # minor
    else:
        return 2  # major

feat_clean["target"] = feat_clean["delay_seconds"].apply(classify_delay)

TARGET_LABELS = {0: "on_time", 1: "minor", 2: "major"}
TARGET_NAMES = ["on_time", "minor", "major"]

# Class distribution
class_dist = feat_clean["target"].value_counts().sort_index()
class_pct = feat_clean["target"].value_counts(normalize=True).sort_index() * 100

print("Target class distribution:")
for cls_id in sorted(class_dist.index):
    print(f"  {TARGET_LABELS[cls_id]:>8s}: {class_dist[cls_id]:>8,} ({class_pct[cls_id]:.1f}%)")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
bars = ax.bar(TARGET_NAMES, class_dist.values, color=colors, edgecolor="none")
for bar, count, pct in zip(bars, class_dist.values, class_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f"{count:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=10)
ax.set_title("Delay Severity Class Distribution", fontweight="bold")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Model Training

We use a temporal train/test split (last 20% of data by timestamp) to avoid data leakage.
An XGBoost classifier handles class imbalance via `scale_pos_weight` adjustments.

In [ ]:
# ---- Temporal Split ----
feat_clean = feat_clean.sort_values("timestamp").reset_index(drop=True)

split_idx = int(len(feat_clean) * 0.8)

FEATURE_COLS = [
    "route_encoded", "stop_encoded",
    "hour", "day_of_week", "is_weekend", "minute_of_day",
    "hour_x_dow", "is_am_rush", "is_pm_rush", "is_rush_hour",
    "hour_sin", "hour_cos",
    "delay_lag_1", "delay_lag_2", "delay_lag_3",
    "delay_rolling_mean_3", "delay_rolling_std_3",
    "alert_active",
]

X_train = feat_clean[FEATURE_COLS].iloc[:split_idx]
X_test = feat_clean[FEATURE_COLS].iloc[split_idx:]
y_train = feat_clean["target"].iloc[:split_idx]
y_test = feat_clean["target"].iloc[split_idx:]

train_end = feat_clean["timestamp"].iloc[split_idx - 1]
test_start = feat_clean["timestamp"].iloc[split_idx]

print(f"Train: {X_train.shape[0]:,} samples ({feat_clean['timestamp'].iloc[0]} to {train_end})")
print(f"Test:  {X_test.shape[0]:,} samples ({test_start} to {feat_clean['timestamp'].iloc[-1]})")

print(f"\nTrain class balance:")
train_dist = y_train.value_counts(normalize=True).sort_index() * 100
for cls_id in sorted(train_dist.index):
    print(f"  {TARGET_LABELS[cls_id]:>8s}: {train_dist[cls_id]:.1f}%")

print(f"\nTest class balance:")
test_dist = y_test.value_counts(normalize=True).sort_index() * 100
for cls_id in sorted(test_dist.index):
    print(f"  {TARGET_LABELS[cls_id]:>8s}: {test_dist[cls_id]:.1f}%")

In [ ]:
# ---- XGBoost Classifier ----
# Compute class weights to handle imbalance
class_counts = y_train.value_counts().sort_index()
total = len(y_train)
n_classes = len(class_counts)
sample_weights = y_train.map(
    {cls: total / (n_classes * count) for cls, count in class_counts.items()}
)

model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    early_stopping_rounds=30,
    random_state=42,
    use_label_encoder=False,
    tree_method="hist",
)

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50,
)

# Training history
results = model.evals_result()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(results["validation_0"]["mlogloss"], label="Train", linewidth=1)
ax.plot(results["validation_1"]["mlogloss"], label="Test", linewidth=1)
ax.set_title("XGBoost Training Progress (Multi-class Log Loss)", fontweight="bold")
ax.set_xlabel("Boosting Round")
ax.set_ylabel("Log Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best iteration: {model.best_iteration}")
print(f"Best test log loss: {model.best_score:.4f}")

In [ ]:
# ---- Evaluation ----
y_pred = model.predict(X_test)

# Classification report
print("Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=TARGET_NAMES, digits=3))

overall_accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {overall_accuracy:.3f}")

# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
cm_normalized = confusion_matrix(y_test, y_pred, normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES,
    ax=axes[0], linewidths=0.5, linecolor="white"
)
axes[0].set_title("Confusion Matrix (Counts)", fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# Normalized
sns.heatmap(
    cm_normalized, annot=True, fmt=".2%", cmap="Blues",
    xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES,
    ax=axes[1], linewidths=0.5, linecolor="white"
)
axes[1].set_title("Confusion Matrix (Normalized)", fontweight="bold")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
# ---- Feature Importance: Top 15 ----
importance = pd.Series(
    model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

# Show top 15
top_15 = importance.tail(15)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#e74c3c" if v == top_15.max() else "#f39c12" if v >= top_15.quantile(0.75) else "steelblue" for v in top_15.values]
top_15.plot(kind="barh", ax=ax, color=colors, edgecolor="none")
ax.set_title("Top 15 Feature Importance (Gain)", fontsize=14, fontweight="bold")
ax.set_xlabel("Feature Importance")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
for feat_name, score in importance.sort_values(ascending=False).head(5).items():
    print(f"  {feat_name}: {score:.4f}")

In [ ]:
# ---- Error Analysis: Per-Route and Per-Hour Accuracy ----
test_df = feat_clean.iloc[split_idx:].copy()
test_df["predicted"] = y_pred
test_df["correct"] = (test_df["target"] == test_df["predicted"]).astype(int)

# Per-route accuracy
route_accuracy = test_df.groupby("route_id").agg(
    accuracy=("correct", "mean"),
    n_samples=("correct", "count"),
    mean_delay=("delay_seconds", "mean"),
    major_pct=("target", lambda x: (x == 2).mean() * 100),
).sort_values("accuracy")

# Per-hour accuracy
hour_accuracy = test_df.groupby("hour").agg(
    accuracy=("correct", "mean"),
    n_samples=("correct", "count"),
    mean_delay=("delay_seconds", "mean"),
).sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Per-route
colors_route = ["#e74c3c" if v < 0.7 else "#f39c12" if v < 0.8 else "#2ecc71" for v in route_accuracy["accuracy"]]
route_accuracy["accuracy"].plot(
    kind="barh", ax=axes[0], color=colors_route, edgecolor="none"
)
axes[0].axvline(x=0.8, color="orange", linestyle="--", alpha=0.7, label="80% threshold")
axes[0].set_title("Model Accuracy by Route", fontweight="bold")
axes[0].set_xlabel("Accuracy")
axes[0].set_ylabel("Route")
axes[0].legend()
axes[0].set_xlim(0, 1)

# Per-hour
ax2 = axes[1]
color_hours = ["#e74c3c" if AM_RUSH_START <= h <= AM_RUSH_END or PM_RUSH_START <= h <= PM_RUSH_END
               else "steelblue" for h in hour_accuracy.index]
ax2.bar(hour_accuracy.index, hour_accuracy["accuracy"], color=color_hours, edgecolor="none")
ax2.axhline(y=0.8, color="orange", linestyle="--", alpha=0.7, label="80% threshold")
ax2.set_title("Model Accuracy by Hour (red = rush hours)", fontweight="bold")
ax2.set_xlabel("Hour of Day")
ax2.set_ylabel("Accuracy")
ax2.set_xticks(range(0, 24))
ax2.set_ylim(0, 1)
ax2.legend()

plt.tight_layout()
plt.show()

# Identify worst-performing segments
print("\nWorst-performing routes (accuracy < 75%):")
worst_routes = route_accuracy[route_accuracy["accuracy"] < 0.75]
if len(worst_routes) > 0:
    for route, row in worst_routes.iterrows():
        print(f"  Route {route}: {row['accuracy']:.1%} accuracy "
              f"(n={row['n_samples']:,.0f}, mean_delay={row['mean_delay']:.0f}s, "
              f"major_pct={row['major_pct']:.1f}%)")
else:
    print("  All routes above 75% accuracy.")

print("\nWorst-performing hours (accuracy < 75%):")
worst_hours = hour_accuracy[hour_accuracy["accuracy"] < 0.75]
if len(worst_hours) > 0:
    for hour, row in worst_hours.iterrows():
        rush_label = " (RUSH)" if AM_RUSH_START <= hour <= AM_RUSH_END or PM_RUSH_START <= hour <= PM_RUSH_END else ""
        print(f"  Hour {hour:02d}:00{rush_label}: {row['accuracy']:.1%} accuracy "
              f"(n={row['n_samples']:,.0f}, mean_delay={row['mean_delay']:.0f}s)")
else:
    print("  All hours above 75% accuracy.")

## Next Steps

This delay prediction model provides a baseline for real-time severity classification. Future work:

1. **Anomaly Detection Notebook** -- Unsupervised detection of unusual delay patterns using
   Isolation Forest and DBSCAN clustering to flag system-wide disruptions before they propagate.

2. **Neural Network Approaches** -- LSTM/Transformer models that process the full sequence of
   delays along a route corridor, capturing propagation effects that lag features approximate.

3. **External Data Integration** -- Weather (precipitation, temperature extremes), MTA planned
   service changes, special events (sports, concerts) as additional predictive features.

4. **Real-Time Scoring Pipeline** -- Deploy the trained model behind an API endpoint that scores
   incoming GTFS-RT updates in real time, feeding predictions into the Railtime frontend.

5. **Model Monitoring** -- Track prediction drift over time as ridership patterns evolve,
   triggering automated retraining when accuracy degrades below threshold.